# Prompt Cache and autotune

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("AZURE_OPENAI_KEY")
api_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_VERSION")


In [16]:
default_temp = 0.2
model_name = "gpt4o"

In [3]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint= api_endpoint,
    azure_deployment="gpt-4o-mini",
    openai_api_key=api_key,
    openai_api_version=api_version,
    temperature=default_temp)

In [4]:
CACHING_DIR ="cache"

In [5]:
from langchain_community.cache import SQLiteCache
from langchain_core.globals import set_llm_cache
set_llm_cache(SQLiteCache(database_path=f"{CACHING_DIR}/lc_cache.sqlite3"))

In [8]:
#convert it to pathlib Path object for easier handling
from pathlib import Path
DISK_CACHE_PATH = Path(f"{CACHING_DIR}/prompt_disk_cache.json")

In [15]:
import json
import hashlib, re

if DISK_CACHE_PATH.exists():
    with open(DISK_CACHE_PATH, "r",encoding="utf-8") as f:
        DISK_CACHE = json.load(f)
else:
    DISK_CACHE = {}

#helper function to trim, collapse multiple whitespaces and lowercase the string for cache freindliness
def normalize(s):
    return re.sub(r'\s+', ' ', s.strip().lower())

#helper function to generate cache key
def generate_cache_key(template, user_input, model, temperature):
    payload = {
        "template": normalize(template),
        "user_input": normalize(user_input),
        "model": model,
        "temperature": temperature
    }
    payload=json.dumps(payload, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()

def cache_get(key):
    return DISK_CACHE.get(key)

def cache_set(key, value):
    DISK_CACHE[key] = value
    with open(DISK_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(DISK_CACHE, f, ensure_ascii=False, indent=2)

In [12]:
PROMPT_VARIANTS = [
    ("You are a helpful assistant.Summarize the ticket into 3 bullet points.Focus on the problem, attempted steps and desired outcome.Avoid fluff."),
     ("You are a support triage assistant. Return exactly 3 bullets: 1)Root cause analysis 2)User impact 3)Recommended next steps. Be concise."
     ),
     ("Act as a senior SRE. Produce 3 bullets under 30 words each. Include: component, symptom and recommended diagnostic step. No extra commentary."),
     ("You are an AI assistant for summarizing support tickets. Extract 3 key bullet points: 1)Issue description 2)Troubleshooting steps taken 3)Expected resolution. Be brief and to the point.")
]

In [13]:
EVAL_SET = [
    "Ticket: User reports that the application crashes when they try to upload a file larger than 5MB. They have attempted to clear their cache and use a different browser, but the issue persists. They expect to be able to upload files up to 20MB without any problems.",
    "Mobile app users are experiencing slow load times and occasional crashes when accessing the dashboard. They have tried reinstalling the app and restarting their devices, but the performance issues continue. They want a smooth and responsive experience when using the dashboard on their mobile devices.",
    "Customer reports that they are unable to reset their password using the 'Forgot Password' feature. They have attempted to use the feature multiple times, but they do not receive the password reset email. They expect to be able to reset their password easily and regain access to their account.",
]

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
#build a chain from templates
def build_chain(template):
    prompt = ChatPromptTemplate.from_messages([
        ("system", template),
        ("human","{ticket}")])
    return prompt | llm | StrOutputParser()

In [17]:
import time
from langchain_community.callbacks import get_openai_callback
def cached_invoke(template, ticket):
    key = generate_cache_key(template, ticket, model_name, default_temp)
    cached = cache_get(key)
    if cached:
        print("Cache hit!")
        return {
            "output":cached,
            "latency_sec":0,
            "tokens":0,
            "source":"disk-cache"
        }
    print("Cache miss. Invoking LLM...")
    chain = build_chain(template)
    start = time.perf_counter()
    with get_openai_callback() as cb:
        out = chain.invoke({"ticket": ticket})
        latency = time.perf_counter() - start
    
    cache_set(key, out)

    result = chain.invoke({"ticket": ticket})
    cache_set(key, result)
    return {"output":result,"latency_sec":latency,"tokens":cb.total_tokens,"source":"llm"}

In [18]:
def summarize_ticket(ticket):
    template = PROMPT_VARIANTS[3] 
    result = cached_invoke(template, ticket)
    return result


In [19]:
summarize_ticket(EVAL_SET[0])

Cache miss. Invoking LLM...


{'output': '- Issue: Application crashes when uploading files larger than 5MB.  \n- Troubleshooting: User cleared cache and tried a different browser; issue remains.  \n- Expected resolution: Enable file uploads up to 20MB without crashing.',
 'latency_sec': 2.3605765000002066,
 'tokens': 159,
 'source': 'llm'}

In [20]:
summarize_ticket(EVAL_SET[0])

Cache hit!


{'output': '- Issue: Application crashes when uploading files larger than 5MB.  \n- Troubleshooting: User cleared cache and tried a different browser; issue remains.  \n- Expected resolution: Enable file uploads up to 20MB without crashing.',
 'latency_sec': 0,
 'tokens': 0,
 'source': 'disk-cache'}